In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from threadpoolctl import threadpool_limits


In [ ]:
# Find the project root whether Jupyter starts here, in the notebook folder, or its parent.
cwd = Path.cwd().resolve()
candidates = [cwd, *cwd.parents]
candidates += [folder / "applied-ml-midterm" for folder in candidates.copy()]
candidates.append(Path("/Users/prith/Documents/Cornell/Applied Machine Learning/midterm/applied-ml-midterm"))
PROJECT_DIR = next((folder for folder in candidates if (folder / "part1-data").is_dir()), None)
if PROJECT_DIR is None:
    raise FileNotFoundError(f"Could not find part1-data from {cwd}")
DATA_DIR = PROJECT_DIR / "part1-data"
OUTPUT_DIR = PROJECT_DIR / "prith_attempts" / "part_1"

part1_train = pd.read_csv(DATA_DIR / "part1_train.csv")
part1_val = pd.read_csv(DATA_DIR / "part1_val.csv")
part1_test = pd.read_csv(DATA_DIR / "part1_test.csv")
REASONS = ["Safety", "Funding", "Efficacy", "Recruitment", "Other"]
EMBEDDINGS = [f"emb_{i}" for i in range(150)]
print("Train, validation, test:", len(part1_train), len(part1_val), len(part1_test))


In [ ]:
def scale_embeddings(train_df, val_df, test_df):
    """Fit the scaler on training embeddings only."""
    scaler = StandardScaler()
    X_train = scaler.fit_transform(train_df[EMBEDDINGS])
    X_val = scaler.transform(val_df[EMBEDDINGS])
    X_test = scaler.transform(test_df[EMBEDDINGS])
    return X_train, X_val, X_test, scaler


def observed_label_mask(train_df):
    known = train_df[REASONS].isin([0, 1]).all(axis=1).to_numpy()
    hidden = train_df[REASONS].eq(-100).all(axis=1).to_numpy()
    if not (known | hidden).all():
        raise ValueError("Expected each training trial to have five observed or five masked labels.")
    return known, hidden


def cluster_reason_scores(X_train, train_df, known, n_clusters=10,
                          smoothing_strength=2.0, random_state=42):
    """K-means on every training embedding; rates use observed labels only."""
    y_known = train_df.loc[known, REASONS].to_numpy(dtype=int)
    model = KMeans(n_clusters=n_clusters, init="k-means++", n_init=10,
                   max_iter=300, tol=1e-4, random_state=random_state)
    with threadpool_limits(limits=1):
        cluster_ids = model.fit_predict(X_train)
    member_counts = np.bincount(cluster_ids[known], minlength=n_clusters)
    positive_counts = np.zeros((n_clusters, len(REASONS)), dtype=float)
    np.add.at(positive_counts, cluster_ids[known], y_known)
    prior = y_known.mean(axis=0)
    rates = (positive_counts + smoothing_strength * prior) / (member_counts[:, None] + smoothing_strength)
    return model, cluster_ids, rates, member_counts


def neighbor_reason_scores(X_train, train_df, known, hidden, n_neighbors=3):
    """Vote among originally observed trials in the same scaled150 space."""
    y_known = train_df.loc[known, REASONS].to_numpy(dtype=int)
    model = KNeighborsClassifier(n_neighbors=n_neighbors, weights="uniform",
                                 metric="euclidean", algorithm="brute", n_jobs=1)
    model.fit(X_train[known], y_known)
    with threadpool_limits(limits=1):
        probabilities = model.predict_proba(X_train[hidden])
    votes = np.column_stack([
        probability[:, np.flatnonzero(classes == 1)[0]]
        for probability, classes in zip(probabilities, model.classes_)
    ])
    return model, votes


def threshold_reasons(scores, threshold):
    """Apply independent multilabel threshold, filling empty rows by argmax."""
    prediction = (np.asarray(scores) >= threshold).astype(int)
    empty = prediction.sum(axis=1) == 0
    prediction[np.flatnonzero(empty), np.asarray(scores)[empty].argmax(axis=1)] = 1
    return prediction


def recover_training_labels(train_df, known, hidden, cluster_ids, cluster_rates,
                            neighbor_votes, threshold=0.30, cluster_weight=0.5):
    """Blend cluster and neighbor scores only for the 1014 masked trials."""
    cluster_scores = cluster_rates[cluster_ids[hidden]]
    scores = cluster_weight * cluster_scores + (1 - cluster_weight) * neighbor_votes
    recovered = threshold_reasons(scores, threshold)
    labels = np.empty((len(train_df), len(REASONS)), dtype=int)
    labels[known] = train_df.loc[known, REASONS].to_numpy(dtype=int)
    labels[hidden] = recovered
    assert np.isin(labels, [0, 1]).all()
    assert np.array_equal(labels[known], train_df.loc[known, REASONS].to_numpy(dtype=int))
    return labels, scores


def fit_reason_models(X_train, labels, known, C=0.1, pseudo_weight=0.25):
    """Five binary logistic regressions; observed rows have weight 1."""
    weights = np.where(known, 1.0, pseudo_weight)
    return [
        LogisticRegression(C=C, max_iter=1000, solver="liblinear", random_state=42)
        .fit(X_train, labels[:, j], sample_weight=weights)
        for j in range(len(REASONS))
    ]


def predict_reasons(models, X):
    probabilities = np.column_stack([model.predict_proba(X)[:, 1] for model in models])
    return threshold_reasons(probabilities, 0.5)


def reason_metrics(truth, prediction):
    return pd.Series({
        "Sample-wise F1": f1_score(truth, prediction, average="samples", zero_division=0),
        "Macro F1": f1_score(truth, prediction, average="macro", zero_division=0),
        **dict(zip(REASONS, f1_score(truth, prediction, average=None, zero_division=0))),
    })


In [ ]:
# Fixed teammate-selected settings; this notebook does not rerun their search grid.
X_train, X_val, X_test, scaler = scale_embeddings(part1_train, part1_val, part1_test)
known, hidden = observed_label_mask(part1_train)
assert known.sum() == 157 and hidden.sum() == 1014
kmeans_model, cluster_ids, cluster_rates, cluster_known_counts = cluster_reason_scores(
    X_train, part1_train, known, n_clusters=10, smoothing_strength=2.0, random_state=42
)
neighbor_model, neighbor_votes = neighbor_reason_scores(
    X_train, part1_train, known, hidden, n_neighbors=3
)
augmented_labels, recovery_scores = recover_training_labels(
    part1_train, known, hidden, cluster_ids, cluster_rates, neighbor_votes,
    threshold=0.30, cluster_weight=0.5
)
reason_models = fit_reason_models(
    X_train, augmented_labels, known, C=0.1, pseudo_weight=0.25
)
validation_predictions = predict_reasons(reason_models, X_val)
validation_metrics = reason_metrics(
    part1_val[REASONS].to_numpy(dtype=int), validation_predictions
)
test_predictions = predict_reasons(reason_models, X_test)
print("Validation metrics:")
print(validation_metrics.to_string())
print("Recovered positives:", dict(zip(REASONS, augmented_labels[hidden].sum(axis=0))))
print("Unresolved training trials:", 0)


In [ ]:
# Export only after checking the fixed method against the saved teammate artifacts.
submission_path = OUTPUT_DIR / "part1_teammate_hybrid_rebuilt_submission.csv"
submission = part1_test[["ID"]].copy()
submission["Predicted"] = [
    " ".join(reason for reason, value in zip(REASONS, row) if value)
    for row in test_predictions
]
assert len(submission) == len(part1_test) == 312
assert submission["ID"].equals(part1_test["ID"])
assert submission["Predicted"].notna().all()
assert submission["Predicted"].str.strip().ne("").all()
submission.to_csv(submission_path, index=False)
print("Saved:", submission_path)
submission.head()
